# 05 Models — TensorFlow — Men's

TensorFlow/Keras neural network trained with **Brier loss** (MSE on probabilities). Same architecture philosophy as the PyTorch model — small network to avoid overfitting on limited tournament data.

**Inputs** (from S3 `04_preprocessing/mens/`):
- `train_features.parquet`, `stage1_features.parquet`, `stage2_features.parquet`, `feature_columns.parquet`

**Outputs** (to S3 `05_models/tensorflow/mens/`):
- `oof_predictions.parquet`, `stage1_predictions.parquet`, `stage2_predictions.parquet`, `cv_results.parquet`

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import brier_score_loss
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression

print(f"TensorFlow version: {tf.__version__}")

#### Functions

In [ ]:
def read_parquet(filename):
    """Read parquet from S3 or local."""
    try:
        return pd.read_parquet(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_parquet(f"../../04_preprocessing/output/{filename}")

In [ ]:
def build_model(n_features, hidden_dim=64, dropout=0.3):
    """Build a simple feedforward Keras model with Brier (MSE) loss."""
    model = keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(hidden_dim, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(hidden_dim // 2, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(1, activation='sigmoid')
    ])
    # Brier score = MSE of predicted probability vs binary outcome
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss='mse',
                  metrics=['mse'])
    return model

#### Constants

In [ ]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "mens"
STAGE = "05_models/tensorflow"
INPUT_PREFIX = f"s3://{BUCKET}/04_preprocessing/{GENDER}/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

LOCAL_OUTPUT = "output/"

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

#### Make output directory

In [ ]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Data

In [ ]:
train = read_parquet("train_features.parquet")
stage1 = read_parquet("stage1_features.parquet")
stage2 = read_parquet("stage2_features.parquet")
feature_cols = read_parquet("feature_columns.parquet")['feature'].tolist()

print(f"Training data: {train.shape}")
print(f"Stage 1 grid: {stage1.shape}")
print(f"Stage 2 grid: {stage2.shape}")
print(f"Features: {len(feature_cols)}")

## 2. LOGO Cross-Validation

In [ ]:
train_original = train[train['TeamA'] < train['TeamB']].copy()
folds = sorted(train['Fold'].unique())

oof_preds = []
cv_results = []

EPOCHS = 200
PATIENCE = 20
BATCH_SIZE = 128

for fold in folds:
    train_mask = train['Fold'] != fold
    val_mask = (train_original['Fold'] == fold)
    
    X_train_raw = train.loc[train_mask, feature_cols].values
    y_train_raw = train.loc[train_mask, 'Label'].values.astype(np.float32)
    X_val_raw = train_original.loc[val_mask, feature_cols].values
    y_val_raw = train_original.loc[val_mask, 'Label'].values.astype(np.float32)
    
    if len(X_val_raw) == 0:
        continue
    
    # Impute and scale
    X_train_imp = np.nan_to_num(X_train_raw, nan=0.0).astype(np.float32)
    X_val_imp = np.nan_to_num(X_val_raw, nan=0.0).astype(np.float32)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_val_scaled = scaler.transform(X_val_imp)
    
    model = build_model(len(feature_cols))
    
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=0
    )
    
    history = model.fit(
        X_train_scaled, y_train_raw,
        validation_data=(X_val_scaled, y_val_raw),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=0
    )
    
    preds = model.predict(X_val_scaled, verbose=0).flatten()
    brier = brier_score_loss(y_val_raw, preds)
    best_epoch = early_stop.stopped_epoch - PATIENCE if early_stop.stopped_epoch > 0 else len(history.history['loss'])
    
    cv_results.append({
        'Fold': fold,
        'BrierScore': brier,
        'Games': len(y_val_raw),
        'BestEpoch': best_epoch
    })
    
    fold_oof = train_original.loc[val_mask, ['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val']].copy()
    fold_oof['Pred'] = preds
    oof_preds.append(fold_oof)
    
    print(f"  Fold {fold}: Brier={brier:.4f}, Games={len(y_val_raw)}, BestEpoch={best_epoch}")

oof_df = pd.concat(oof_preds, ignore_index=True)
cv_df = pd.DataFrame(cv_results)

In [ ]:
overall_brier = brier_score_loss(oof_df['Label'], oof_df['Pred'])
stage1_oof = oof_df[oof_df['IsStage1Val'] == 1]
stage1_brier = brier_score_loss(stage1_oof['Label'], stage1_oof['Pred']) if len(stage1_oof) > 0 else None

print(f"\nOverall OOF Brier Score: {overall_brier:.4f}")
print(f"Stage 1 (2022-2025) Brier Score: {stage1_brier:.4f}" if stage1_brier else "No Stage 1 data")
print(f"Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")

## 3. Train Final Model and Calibrate

In [ ]:
# Fit scaler on all training data
X_all = np.nan_to_num(train[feature_cols].values, nan=0.0).astype(np.float32)
y_all = train['Label'].values.astype(np.float32)

final_scaler = StandardScaler()
X_all_scaled = final_scaler.fit_transform(X_all)

# Train final model with validation split for early stopping
final_model = build_model(len(feature_cols))
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=0
)

final_model.fit(
    X_all_scaled, y_all,
    validation_split=0.1,
    epochs=300,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=0
)

# Calibration
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(oof_df['Pred'].values, oof_df['Label'].values)

oof_df['PredCalibrated'] = calibrator.predict(oof_df['Pred'].values)
calibrated_brier = brier_score_loss(oof_df['Label'], oof_df['PredCalibrated'])
print(f"OOF Brier (raw): {overall_brier:.4f}")
print(f"OOF Brier (calibrated): {calibrated_brier:.4f}")

## 4. Generate Predictions

In [ ]:
X_stage1 = final_scaler.transform(np.nan_to_num(stage1[feature_cols].values, nan=0.0).astype(np.float32))
stage1['Pred_tensorflow'] = final_model.predict(X_stage1, verbose=0).flatten()
stage1['Pred_tensorflow_calibrated'] = calibrator.predict(stage1['Pred_tensorflow'].values).clip(0.02, 0.98)

X_stage2 = final_scaler.transform(np.nan_to_num(stage2[feature_cols].values, nan=0.0).astype(np.float32))
stage2['Pred_tensorflow'] = final_model.predict(X_stage2, verbose=0).flatten()
stage2['Pred_tensorflow_calibrated'] = calibrator.predict(stage2['Pred_tensorflow'].values).clip(0.02, 0.98)

print(f"Stage 1 predictions range: [{stage1['Pred_tensorflow_calibrated'].min():.3f}, {stage1['Pred_tensorflow_calibrated'].max():.3f}]")
print(f"Stage 2 predictions range: [{stage2['Pred_tensorflow_calibrated'].min():.3f}, {stage2['Pred_tensorflow_calibrated'].max():.3f}]")

stage1_actual = stage1.dropna(subset=['Label'])
if len(stage1_actual) > 0:
    s1_brier = brier_score_loss(stage1_actual['Label'], stage1_actual['Pred_tensorflow_calibrated'])
    print(f"Stage 1 Brier (calibrated): {s1_brier:.4f}")

## 5. Save Outputs

In [ ]:
outputs = {
    'oof_predictions': oof_df[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val', 'Pred', 'PredCalibrated']],
    'stage1_predictions': stage1[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Pred_tensorflow', 'Pred_tensorflow_calibrated']],
    'stage2_predictions': stage2[['ID', 'Season', 'TeamA', 'TeamB', 'Pred_tensorflow', 'Pred_tensorflow_calibrated']],
    'cv_results': cv_df,
}

for name, df in outputs.items():
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

## 6. Summary

In [ ]:
print("=" * 60)
print("TENSORFLOW MODEL SUMMARY — MEN'S")
print("=" * 60)
print(f"\nOOF Brier Score (raw): {overall_brier:.4f}")
print(f"OOF Brier Score (calibrated): {calibrated_brier:.4f}")
print(f"CV Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")
print(f"Architecture: Dense(64) -> Dense(32) -> Dense(1) with MSE (Brier) loss")